# Présentation du notebook

Ce notebook présente l'étude de la génération automatique de requêtes AQL à partir de questions formulées en langage naturel à l'aide de modèles de langage (LLM).

# Objectif expérimental

L'objectif est d'analyser la capacité des modèles à traduire une intention utilisateur en une requête AQL valide en étudiant différentes stratégies de prompting.

# Stratégies de prompting étudiées

- Direct Prompting : génération sans contexte détaillé du schéma.
- Schema Prompting : ajout des informations relatives à la structure de la base.
- Few-shot Prompting : utilisation d'exemples question/requête pour guider le modèle.

In [16]:
import json
import time
import re
import requests
import pandas as pd
from pathlib import Path

In [17]:
# Détection automatique de la racine du projet
current = Path.cwd().resolve()

if (current / "arangodb").exists() and (current / "data").exists():
    PROJECT_ROOT = current
elif (current.parent / "arangodb").exists() and (current.parent / "data").exists():
    PROJECT_ROOT = current.parent
else:
    raise FileNotFoundError("Impossible de détecter la racine du projet.")

BENCHMARK_PATH = PROJECT_ROOT / "data" / "benchmark" / "yelp_benchmark.json"
SCHEMA_PATH = PROJECT_ROOT / "arangodb" / "schema_description.json"
RESULTS_PATH = PROJECT_ROOT / "results" / "generated_queries.csv"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("Benchmark :", BENCHMARK_PATH)
print("Schema :", SCHEMA_PATH)

assert BENCHMARK_PATH.exists(), "Benchmark introuvable"
assert SCHEMA_PATH.exists(), "Schéma introuvable"

print("Tous les chemins sont valides.")

PROJECT_ROOT : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project
Benchmark : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\data\benchmark\yelp_benchmark.json
Schema : C:\Users\hp\Desktop\bureau\s8\LLM_Yelp_Project\arangodb\schema_description.json
Tous les chemins sont valides.


In [18]:
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema = json.load(f)

print("Questions chargées :", len(benchmark))
print("Collections document :", len(schema.get("document_collections", [])))
print("Collections edge :", len(schema.get("edge_collections", [])))

Questions chargées : 128
Collections document : 7
Collections edge : 7


In [19]:
def build_direct_prompt(question):
    return f"""
You are an expert in ArangoDB AQL.

Convert the following natural language question into one valid AQL query.

Return ONLY the AQL query.
Do not provide explanations.
Do not use SQL.
Do not use INSERT, UPDATE, REMOVE or REPLACE.

Question:
{question}

AQL:
""".strip()

In [20]:
question = benchmark[0]["question"]

prompt = build_direct_prompt(question)

print("Question :", question)
print()
print(prompt)

Question : Give me all the moroccan restaurants in Texas

You are an expert in ArangoDB AQL.

Convert the following natural language question into one valid AQL query.

Return ONLY the AQL query.
Do not provide explanations.
Do not use SQL.
Do not use INSERT, UPDATE, REMOVE or REPLACE.

Question:
Give me all the moroccan restaurants in Texas

AQL:


In [27]:
import requests
import time

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "mistral:latest"

def generate_aql(prompt):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }

    start_time = time.time()

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=900
    )

    response.raise_for_status()

    generation_time = time.time() - start_time

    result = response.json()

    return result["response"].strip(), generation_time

In [28]:
question = benchmark[0]["question"]

prompt = build_direct_prompt(question)

generated_aql, generation_time = generate_aql(prompt)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Mistral :")
print(generated_aql)

print("\nTemps de génération :", round(generation_time, 2), "secondes")

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Mistral :
```
FOR doc IN collectionname
FILTER doc.cuisine == "Moroccan" AND doc.location == "Texas"
RETURN doc
```

Temps de génération : 11.93 secondes


In [29]:
direct_results = []

direct_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": generated_aql,
    "model": MODEL_NAME,
    "strategy": "direct",
    "generation_time": round(generation_time, 2)
})

direct_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': '```\nFOR doc IN collectionname\nFILTER doc.cuisine == "Moroccan" AND doc.location == "Texas"\nRETURN doc\n```',
 'model': 'mistral:latest',
 'strategy': 'direct',
 'generation_time': 11.93}

In [30]:
def build_full_schema_prompt(question, schema):
    return f"""
You are an expert in ArangoDB AQL.

Convert the following natural language question into one valid AQL query.

Use ONLY the collections, attributes and relations provided in the schema below.

Return ONLY the AQL query.
Do not provide explanations.
Do not use SQL.
Do not use INSERT, UPDATE, REMOVE or REPLACE.

Schema:
{json.dumps(schema, indent=2, ensure_ascii=False)}

Question:
{question}

AQL:
""".strip()

In [31]:
question = benchmark[0]["question"]

prompt_schema = build_full_schema_prompt(
    question,
    schema
)

generated_aql_schema, generation_time_schema = generate_aql(
    prompt_schema
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée - schéma complet :")
print(generated_aql_schema)

print(
    "\nTemps de génération :",
    round(generation_time_schema, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée - schéma complet :
```
FOR v, e, p IN 1..100000
  FILTER p.BusinessNeighborhood.to.state == "Texas" && v._collection == "Businesses" && ANY v.Categories.category_name IN ["Moroccan Restaurant"]
  RETURN {
    business_id: v.business_id,
    name: v.name,
    city: v.city,
    state: v.state
  }
```

Temps de génération : 65.18 secondes


In [32]:
schema_results = []

schema_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": generated_aql_schema,
    "model": MODEL_NAME,
    "strategy": "full_schema",
    "generation_time": round(generation_time_schema, 2)
})

schema_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': '```\nFOR v, e, p IN 1..100000\n  FILTER p.BusinessNeighborhood.to.state == "Texas" && v._collection == "Businesses" && ANY v.Categories.category_name IN ["Moroccan Restaurant"]\n  RETURN {\n    business_id: v.business_id,\n    name: v.name,\n    city: v.city,\n    state: v.state\n  }\n```',
 'model': 'mistral:latest',
 'strategy': 'full_schema',
 'generation_time': 65.18}

In [33]:
REDUCED_SCHEMA = """
DOCUMENT COLLECTIONS

Businesses:
- bid
- business_id
- city
- full_address
- is_open
- latitude
- longitude
- name
- rating
- review_count
- state

Categories:
- business_id
- category_name
- id

Reviews:
- business_id
- month
- rating
- rid
- text
- user_id
- year

Tips:
- business_id
- likes
- month
- text
- tip_id
- user_id
- year

Checkins:
- business_id
- cid
- count
- day

Neighborhoods:
- business_id
- id
- neighborhood_name

Users:
- name
- uid
- user_id


EDGE COLLECTIONS

WritesReview:
Users -> Reviews

ReviewsBusiness:
Reviews -> Businesses

WritesTip:
Users -> Tips

TipsBusiness:
Tips -> Businesses

BusinessCategory:
Businesses -> Categories

BusinessCheckin:
Businesses -> Checkins

BusinessNeighborhood:
Businesses -> Neighborhoods
""".strip()

In [34]:
def build_reduced_schema_prompt(question):
    return f"""
You are an expert in ArangoDB AQL.

Convert the following natural language question into one valid AQL query.

Use ONLY the collections, attributes and relations provided below.

Return ONLY the AQL query.
Do not provide explanations.
Do not use SQL.
Do not use INSERT, UPDATE, REMOVE or REPLACE.

Schema:
{REDUCED_SCHEMA}

Question:
{question}

AQL:
""".strip()

In [35]:
question = benchmark[0]["question"]

prompt_reduced = build_reduced_schema_prompt(
    question
)

generated_aql_reduced, generation_time_reduced = generate_aql(
    prompt_reduced
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée - schéma réduit :")
print(generated_aql_reduced)

print(
    "\nTemps de génération :",
    round(generation_time_reduced, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée - schéma réduit :
```
FOR b IN Businesses
FILTER b.name == "Moroccan" AND b.state == "Texas"
FOR c IN BusinessNeighborhoods FILTER c.business_id == b._id
FOR n IN Neighborhoods FILTER n._id == c.id
FILTER n.neighborhood_name == "Texas"
FOR cat IN BusinessCategory FILTER cat.business_id == b._id
FOR cat_name IN Categories FILTER cat_name._id == cat.category_id
FILTER cat_name.category_name == "Restaurant"
RETURN b
```

Temps de génération : 110.89 secondes


In [36]:
reduced_schema_results = []

reduced_schema_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": generated_aql_reduced,
    "model": MODEL_NAME,
    "strategy": "reduced_schema",
    "generation_time": round(generation_time_reduced, 2)
})

reduced_schema_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': '```\nFOR b IN Businesses\nFILTER b.name == "Moroccan" AND b.state == "Texas"\nFOR c IN BusinessNeighborhoods FILTER c.business_id == b._id\nFOR n IN Neighborhoods FILTER n._id == c.id\nFILTER n.neighborhood_name == "Texas"\nFOR cat IN BusinessCategory FILTER cat.business_id == b._id\nFOR cat_name IN Categories FILTER cat_name._id == cat.category_id\nFILTER cat_name.category_name == "Restaurant"\nRETURN b\n```',
 'model': 'mistral:latest',
 'strategy': 'reduced_schema',
 'generation_time': 110.89}

In [37]:
FEW_SHOT_IDS = ["Q002", "Q004", "Q013"]

few_shot_examples = [
    item for item in benchmark
    if item["id"] in FEW_SHOT_IDS
]

for example in few_shot_examples:
    print("ID :", example["id"])
    print("Question :", example["question"])
    print("AQL :")
    print(example["gold_aql"])
    print("-" * 70)

ID : Q002
Question : List all the Italian restaurants in Los Angeles
AQL :

FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Italian"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }

----------------------------------------------------------------------
ID : Q004
Question : List all the restaurants rated more than 3.5
AQL :

FOR b IN Businesses
    FILTER b.rating > 3.5

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }

----------------------------------------------------------------------
ID : Q013
Question : list all the reviews by Niloofar
AQL :

FOR u IN Users
    FILTER u.name == "Niloof

In [38]:
def build_few_shot_prompt(question, examples):

    examples_text = ""

    for example in examples:
        examples_text += f"""
Question:
{example["question"]}

AQL:
{example["gold_aql"]}

"""

    return f"""
You are an expert in ArangoDB AQL.

Convert the natural language question into one valid AQL query.

Use the following examples as demonstrations.

{examples_text}

Return ONLY the AQL query.
Do not provide explanations.
Do not use SQL.
Do not use INSERT, UPDATE, REMOVE or REPLACE.

Question:
{question}

AQL:
""".strip()

In [39]:
question = benchmark[0]["question"]

prompt_few_shot = build_few_shot_prompt(
    question,
    few_shot_examples
)

print(prompt_few_shot)

You are an expert in ArangoDB AQL.

Convert the natural language question into one valid AQL query.

Use the following examples as demonstrations.


Question:
List all the Italian restaurants in Los Angeles

AQL:

FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Italian"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }



Question:
List all the restaurants rated more than 3.5

AQL:

FOR b IN Businesses
    FILTER b.rating > 3.5

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }



Question:
list all the reviews by Niloofar

AQL:

FOR u IN Users
    FILTER u.name == "Niloofar"

    FOR r IN 1..1

In [40]:
question = benchmark[0]["question"]

prompt_few_shot = build_few_shot_prompt(
    question,
    few_shot_examples
)

print(prompt_few_shot)

You are an expert in ArangoDB AQL.

Convert the natural language question into one valid AQL query.

Use the following examples as demonstrations.


Question:
List all the Italian restaurants in Los Angeles

AQL:

FOR b IN Businesses
    FILTER b.city == "Los Angeles"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Italian"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }



Question:
List all the restaurants rated more than 3.5

AQL:

FOR b IN Businesses
    FILTER b.rating > 3.5

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }



Question:
list all the reviews by Niloofar

AQL:

FOR u IN Users
    FILTER u.name == "Niloofar"

    FOR r IN 1..1

In [41]:
generated_aql_few_shot, generation_time_few_shot = generate_aql(
    prompt_few_shot
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée - Few-shot :")
print(generated_aql_few_shot)

print(
    "\nTemps de génération :",
    round(generation_time_few_shot, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée - Few-shot :
FOR b IN Businesses
    FILTER b.city == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Restaurants" AND c.sub_category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }

Temps de génération : 106.05 secondes


In [42]:
few_shot_results = []

few_shot_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": generated_aql_few_shot,
    "model": MODEL_NAME,
    "strategy": "few_shot",
    "generation_time": round(generation_time_few_shot, 2)
})

few_shot_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': 'FOR b IN Businesses\n    FILTER b.city == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Restaurants" AND c.sub_category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            rating: b.rating\n        }',
 'model': 'mistral:latest',
 'strategy': 'few_shot',
 'generation_time': 106.05}

# QWEN

In [43]:
MODEL_NAME = "qwen3:8b"

print("Modèle utilisé :", MODEL_NAME)

Modèle utilisé : qwen3:8b


In [44]:
question = benchmark[0]["question"]

prompt_qwen_direct = build_direct_prompt(question)

qwen_direct_aql, qwen_direct_time = generate_aql(
    prompt_qwen_direct
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Qwen3 8B - Direct :")
print(qwen_direct_aql)

print(
    "\nTemps de génération :",
    round(qwen_direct_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Qwen3 8B - Direct :
FOR r IN restaurants
FILTER r.country == 'Morocco' AND r.state == 'Texas'
RETURN r

Temps de génération : 257.68 secondes


In [45]:
qwen_direct_results = []

qwen_direct_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": qwen_direct_aql,
    "model": MODEL_NAME,
    "strategy": "direct",
    "generation_time": round(qwen_direct_time, 2)
})

qwen_direct_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': "FOR r IN restaurants\nFILTER r.country == 'Morocco' AND r.state == 'Texas'\nRETURN r",
 'model': 'qwen3:8b',
 'strategy': 'direct',
 'generation_time': 257.68}

In [46]:
question = benchmark[0]["question"]

prompt_qwen_full_schema = build_full_schema_prompt(
    question,
    schema
)

qwen_full_schema_aql, qwen_full_schema_time = generate_aql(
    prompt_qwen_full_schema
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Qwen3 8B - Schéma complet :")
print(qwen_full_schema_aql)

print(
    "\nTemps de génération :",
    round(qwen_full_schema_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Qwen3 8B - Schéma complet :
FOR business IN Businesses
  FILTER business.state == "Texas"
  FILTER EXISTS(
    FOR category IN BusinessCategory
      FILTER category._from == business._id
      FILTER DOCUMENT(category._to).category_name == "Moroccan"
      RETURN true
  )
  RETURN business

Temps de génération : 845.15 secondes


In [47]:
qwen_full_schema_results = []

qwen_full_schema_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": qwen_full_schema_aql,
    "model": MODEL_NAME,
    "strategy": "full_schema",
    "generation_time": round(qwen_full_schema_time, 2)
})

qwen_full_schema_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': 'FOR business IN Businesses\n  FILTER business.state == "Texas"\n  FILTER EXISTS(\n    FOR category IN BusinessCategory\n      FILTER category._from == business._id\n      FILTER DOCUMENT(category._to).category_name == "Moroccan"\n      RETURN true\n  )\n  RETURN business',
 'model': 'qwen3:8b',
 'strategy': 'full_schema',
 'generation_time': 845.15}

In [48]:
question = benchmark[0]["question"]

prompt_qwen_reduced = build_reduced_schema_prompt(
    question
)

qwen_reduced_aql, qwen_reduced_time = generate_aql(
    prompt_qwen_reduced
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Qwen3 8B - Schéma réduit :")
print(qwen_reduced_aql)

print(
    "\nTemps de génération :",
    round(qwen_reduced_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Qwen3 8B - Schéma réduit :
FOR business IN Businesses
  FILTER business.state == 'Texas'
  FILTER EXISTS (
    FOR edge IN BusinessCategory
      FILTER edge._from == business._id
      LET category = DOCUMENT(edge._to)
      FILTER category.category_name == 'Moroccan'
      RETURN true
  )
  RETURN business

Temps de génération : 516.48 secondes


In [49]:
qwen_reduced_results = []

qwen_reduced_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": qwen_reduced_aql,
    "model": MODEL_NAME,
    "strategy": "reduced_schema",
    "generation_time": round(qwen_reduced_time, 2)
})

qwen_reduced_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': "FOR business IN Businesses\n  FILTER business.state == 'Texas'\n  FILTER EXISTS (\n    FOR edge IN BusinessCategory\n      FILTER edge._from == business._id\n      LET category = DOCUMENT(edge._to)\n      FILTER category.category_name == 'Moroccan'\n      RETURN true\n  )\n  RETURN business",
 'model': 'qwen3:8b',
 'strategy': 'reduced_schema',
 'generation_time': 516.48}

In [50]:
question = benchmark[0]["question"]

prompt_qwen_few_shot = build_few_shot_prompt(
    question,
    few_shot_examples
)

qwen_few_shot_aql, qwen_few_shot_time = generate_aql(
    prompt_qwen_few_shot
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Qwen3 8B - Few-shot :")
print(qwen_few_shot_aql)

print(
    "\nTemps de génération :",
    round(qwen_few_shot_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Qwen3 8B - Few-shot :
FOR b IN Businesses
    FILTER b.city == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }

Temps de génération : 214.96 secondes


In [51]:
qwen_few_shot_results = []

qwen_few_shot_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": qwen_few_shot_aql,
    "model": MODEL_NAME,
    "strategy": "few_shot",
    "generation_time": round(qwen_few_shot_time, 2)
})

qwen_few_shot_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': 'FOR b IN Businesses\n    FILTER b.city == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            rating: b.rating\n        }',
 'model': 'qwen3:8b',
 'strategy': 'few_shot',
 'generation_time': 214.96}

# Gemma3 8b

In [56]:
MODEL_NAME = "gemma3:4b"
question = benchmark[0]["question"]

direct_prompt = build_direct_prompt(question)

print("Question :", question)
print()
print(direct_prompt)

Question : Give me all the moroccan restaurants in Texas

You are an expert in ArangoDB AQL.

Convert the following natural language question into one valid AQL query.

Return ONLY the AQL query.
Do not provide explanations.
Do not use SQL.
Do not use INSERT, UPDATE, REMOVE or REPLACE.

Question:
Give me all the moroccan restaurants in Texas

AQL:


In [57]:
MODEL_NAME = "gemma3:4b"

gemma_direct_aql, gemma_direct_time = generate_aql(direct_prompt)

print("GEMMA3 4B - DIRECT")
print()
print(gemma_direct_aql)
print()
print("Temps :", round(gemma_direct_time, 2), "secondes")

GEMMA3 4B - DIRECT

```aql
FOR doc IN restaurants
  FILTER doc.cuisine == "Moroccan" AND doc.state == "Texas"
  RETURN doc
```

Temps : 29.63 secondes


In [58]:
gemma_direct_result = {
    "strategy": "Direct",
    "model": "gemma3:4b",
    "question": question,
    "generated_aql": gemma_direct_aql,
    "generation_time": gemma_direct_time,
    "correct": False,
    "remarks": "Hallucinated collection 'restaurants' and attribute 'cuisine'."
}

In [60]:
question = benchmark[0]["question"]

prompt_gemma_full_schema = build_full_schema_prompt(
    question,
    schema
)

gemma_full_schema_aql, gemma_full_schema_time = generate_aql(
    prompt_gemma_full_schema
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Gemma3 4B - Schéma complet :")
print(gemma_full_schema_aql)

print(
    "\nTemps de génération :",
    round(gemma_full_schema_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Gemma3 4B - Schéma complet :
```AQL
FOR doc IN YelpDB.Businesses
  FILTER doc.city == "Pittsburgh" AND doc.state == "Pennsylvania"
RETURN doc
```

Temps de génération : 156.03 secondes


In [64]:
gemma_full_schema_results = []

gemma_full_schema_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": gemma_full_schema_aql,
    "model": MODEL_NAME,
    "strategy": "full_schema",
    "generation_time": round(gemma_full_schema_time, 2)
})

gemma_full_schema_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': '```AQL\nFOR doc IN YelpDB.Businesses\n  FILTER doc.city == "Pittsburgh" AND doc.state == "Pennsylvania"\nRETURN doc\n```',
 'model': 'gemma3:4b',
 'strategy': 'full_schema',
 'generation_time': 156.03}

In [63]:
question = benchmark[0]["question"]

prompt_gemma_reduced = build_reduced_schema_prompt(
    question
)

gemma_reduced_aql, gemma_reduced_time = generate_aql(
    prompt_gemma_reduced
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Gemma3 4B - Schéma réduit :")
print(gemma_reduced_aql)

print(
    "\nTemps de génération :",
    round(gemma_reduced_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Gemma3 4B - Schéma réduit :
```aql
FOR business IN Businesses
    FILTER business.city == "Texas" AND business.name CONTAINS "Moroccan"
RETURN business
```

Temps de génération : 50.23 secondes


In [65]:
gemma_reduced_results = []

gemma_reduced_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": gemma_reduced_aql,
    "model": MODEL_NAME,
    "strategy": "reduced_schema",
    "generation_time": round(gemma_reduced_time, 2)
})

gemma_reduced_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': '```aql\nFOR business IN Businesses\n    FILTER business.city == "Texas" AND business.name CONTAINS "Moroccan"\nRETURN business\n```',
 'model': 'gemma3:4b',
 'strategy': 'reduced_schema',
 'generation_time': 50.23}

In [66]:
MODEL_NAME = "gemma3:4b"

question = benchmark[0]["question"]

prompt_gemma_few_shot = build_few_shot_prompt(
    question,
    few_shot_examples
)

gemma_few_shot_aql, gemma_few_shot_time = generate_aql(
    prompt_gemma_few_shot
)

print("Question :")
print(question)

print("\nAQL Gold :")
print(benchmark[0]["gold_aql"])

print("\nAQL générée par Gemma3 4B - Few-shot :")
print(gemma_few_shot_aql)

print(
    "\nTemps de génération :",
    round(gemma_few_shot_time, 2),
    "secondes"
)

Question :
Give me all the moroccan restaurants in Texas

AQL Gold :

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            state: b.state,
            rating: b.rating
        }


AQL générée par Gemma3 4B - Few-shot :
```aql
FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }
```

Temps de génération : 40.84 secondes


In [67]:
gemma_few_shot_results = []

gemma_few_shot_results.append({
    "id": benchmark[0]["id"],
    "question": question,
    "gold_aql": benchmark[0]["gold_aql"],
    "generated_aql": gemma_few_shot_aql,
    "model": MODEL_NAME,
    "strategy": "few_shot",
    "generation_time": round(gemma_few_shot_time, 2),
    "status": "correct_but_incomplete",
    "remarks": "Correct filtering and traversal; state field missing from returned object."
})

gemma_few_shot_results[0]

{'id': 'Q001',
 'question': 'Give me all the moroccan restaurants in Texas',
 'gold_aql': '\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            state: b.state,\n            rating: b.rating\n        }\n',
 'generated_aql': '```aql\nFOR b IN Businesses\n    FILTER b.state == "Texas"\n\n    FOR c IN 1..1 OUTBOUND b BusinessCategory\n        FILTER c.category_name == "Moroccan"\n\n        RETURN DISTINCT {\n            business_id: b.business_id,\n            name: b.name,\n            city: b.city,\n            rating: b.rating\n        }\n```',
 'model': 'gemma3:4b',
 'strategy': 'few_shot',
 'generation_time': 40.84,
 'status': 'correct_but_incomplete',
 'remarks': 'Correct filtering and traversal; state field missing from returned object.'}

In [52]:
import pandas as pd

comparison_results = []

# Mistral
comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "mistral:latest",
    "strategy": "direct",
    "generated_aql": direct_results[0]["generated_aql"],
    "generation_time": direct_results[0]["generation_time"]
})

comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "mistral:latest",
    "strategy": "full_schema",
    "generated_aql": schema_results[0]["generated_aql"],
    "generation_time": schema_results[0]["generation_time"]
})

comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "mistral:latest",
    "strategy": "reduced_schema",
    "generated_aql": reduced_schema_results[0]["generated_aql"],
    "generation_time": reduced_schema_results[0]["generation_time"]
})

comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "mistral:latest",
    "strategy": "few_shot",
    "generated_aql": few_shot_results[0]["generated_aql"],
    "generation_time": few_shot_results[0]["generation_time"]
})

# Qwen3 8B
comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "qwen3:8b",
    "strategy": "direct",
    "generated_aql": qwen_direct_results[0]["generated_aql"],
    "generation_time": qwen_direct_results[0]["generation_time"]
})

comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "qwen3:8b",
    "strategy": "full_schema",
    "generated_aql": qwen_full_schema_results[0]["generated_aql"],
    "generation_time": qwen_full_schema_results[0]["generation_time"]
})

comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "qwen3:8b",
    "strategy": "reduced_schema",
    "generated_aql": qwen_reduced_results[0]["generated_aql"],
    "generation_time": qwen_reduced_results[0]["generation_time"]
})

comparison_results.append({
    "id": benchmark[0]["id"],
    "question": benchmark[0]["question"],
    "model": "qwen3:8b",
    "strategy": "few_shot",
    "generated_aql": qwen_few_shot_results[0]["generated_aql"],
    "generation_time": qwen_few_shot_results[0]["generation_time"]
})

df_comparison = pd.DataFrame(comparison_results)

df_comparison

,id,question,model,strategy,generated_aql,generation_time
0,Q001,Give me all the moroccan restaurants in Texas,mistral:latest,direct,```\nFOR doc IN collectionname\nFILTER doc.cui...,11.93
1,Q001,Give me all the moroccan restaurants in Texas,mistral:latest,full_schema,"```\nFOR v, e, p IN 1..100000\n FILTER p.Busi...",65.18
2,Q001,Give me all the moroccan restaurants in Texas,mistral:latest,reduced_schema,"```\nFOR b IN Businesses\nFILTER b.name == ""Mo...",110.89
3,Q001,Give me all the moroccan restaurants in Texas,mistral:latest,few_shot,"FOR b IN Businesses\n FILTER b.city == ""Tex...",106.05
4,Q001,Give me all the moroccan restaurants in Texas,qwen3:8b,direct,FOR r IN restaurants\nFILTER r.country == 'Mor...,257.68
5,Q001,Give me all the moroccan restaurants in Texas,qwen3:8b,full_schema,FOR business IN Businesses\n FILTER business....,845.15
6,Q001,Give me all the moroccan restaurants in Texas,qwen3:8b,reduced_schema,FOR business IN Businesses\n FILTER business....,516.48
7,Q001,Give me all the moroccan restaurants in Texas,qwen3:8b,few_shot,"FOR b IN Businesses\n FILTER b.city == ""Tex...",214.96


In [69]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

comparison_results = [

    # =========================
    # MISTRAL
    # =========================
    {
        "id": "Q001",
        "model": "mistral:latest",
        "strategy": "direct",
        "generated_aql": direct_results[0]["generated_aql"],
        "generation_time": direct_results[0]["generation_time"],
        "remarks": "Incorrect - strong schema hallucination"
    },
    {
        "id": "Q001",
        "model": "mistral:latest",
        "strategy": "full_schema",
        "generated_aql": schema_results[0]["generated_aql"],
        "generation_time": schema_results[0]["generation_time"],
        "remarks": "Incorrect - invalid traversal and schema usage"
    },
    {
        "id": "Q001",
        "model": "mistral:latest",
        "strategy": "reduced_schema",
        "generated_aql": reduced_schema_results[0]["generated_aql"],
        "generation_time": reduced_schema_results[0]["generation_time"],
        "remarks": "Incorrect - better schema names but wrong relations"
    },
    {
        "id": "Q001",
        "model": "mistral:latest",
        "strategy": "few_shot",
        "generated_aql": few_shot_results[0]["generated_aql"],
        "generation_time": few_shot_results[0]["generation_time"],
        "remarks": "Incorrect - close to Gold but city/state confusion"
    },

    # =========================
    # QWEN3 8B
    # =========================
    {
        "id": "Q001",
        "model": "qwen3:8b",
        "strategy": "direct",
        "generated_aql": qwen_direct_results[0]["generated_aql"],
        "generation_time": qwen_direct_results[0]["generation_time"],
        "remarks": "Incorrect - schema hallucination"
    },
    {
        "id": "Q001",
        "model": "qwen3:8b",
        "strategy": "full_schema",
        "generated_aql": qwen_full_schema_results[0]["generated_aql"],
        "generation_time": qwen_full_schema_results[0]["generation_time"],
        "remarks": "Execution failed - invalid use of EXISTS()"
    },
    {
        "id": "Q001",
        "model": "qwen3:8b",
        "strategy": "reduced_schema",
        "generated_aql": qwen_reduced_results[0]["generated_aql"],
        "generation_time": qwen_reduced_results[0]["generation_time"],
        "remarks": "Execution failed - invalid use of EXISTS()"
    },
    {
        "id": "Q001",
        "model": "qwen3:8b",
        "strategy": "few_shot",
        "generated_aql": qwen_few_shot_results[0]["generated_aql"],
        "generation_time": qwen_few_shot_results[0]["generation_time"],
        "remarks": "Almost correct - city used instead of state"
    },

    # =========================
    # GEMMA3 4B
    # =========================
    {
        "id": "Q001",
        "model": "gemma3:4b",
        "strategy": "direct",
        "generated_aql": gemma_direct_aql,
        "generation_time": round(gemma_direct_time, 2),
        "remarks": "Incorrect - hallucinated collection 'restaurants' and attribute 'cuisine'"
    },
    {
        "id": "Q001",
        "model": "gemma3:4b",
        "strategy": "full_schema",
        "generated_aql": gemma_full_schema_aql,
        "generation_time": round(gemma_full_schema_time, 2),
        "remarks": "Incorrect - wrong collection reference, invented location and Moroccan category ignored"
    },
    {
        "id": "Q001",
        "model": "gemma3:4b",
        "strategy": "reduced_schema",
        "generated_aql": gemma_reduced_aql,
        "generation_time": round(gemma_reduced_time, 2),
        "remarks": "To evaluate"
    },
    {
        "id": "Q001",
        "model": "gemma3:4b",
        "strategy": "few_shot",
        "generated_aql": gemma_few_shot_aql,
        "generation_time": round(gemma_few_shot_time, 2),
        "remarks": "Correct but incomplete - state missing from returned object"
    }
]

df_comparison = pd.DataFrame(comparison_results)

df_comparison

,id,model,strategy,generated_aql,generation_time,remarks
0,Q001,mistral:latest,direct,"```\nFOR doc IN collectionname\nFILTER doc.cuisine == ""Moroccan"" AND doc.location == ""Texas""\nRETURN doc\n```",11.93,Incorrect - strong schema hallucination
1,Q001,mistral:latest,full_schema,"```\nFOR v, e, p IN 1..100000\n FILTER p.BusinessNeighborhood.to.state == ""Texas"" && v._collection == ""Businesses"" && ANY v.Categories.category_name IN [""Moroccan Restaurant""]\n RETURN {\n business_id: v.business_id,\n name: v.name,\n city: v.city,\n state: v.state\n }\n```",65.18,Incorrect - invalid traversal and schema usage
2,Q001,mistral:latest,reduced_schema,"```\nFOR b IN Businesses\nFILTER b.name == ""Moroccan"" AND b.state == ""Texas""\nFOR c IN BusinessNeighborhoods FILTER c.business_id == b._id\nFOR n IN Neighborhoods FILTER n._id == c.id\nFILTER n.neighborhood_name == ""Texas""\nFOR cat IN BusinessCategory FILTER cat.business_id == b._id\nFOR cat_name IN Categories FILTER cat_name._id == cat.category_id\nFILTER cat_name.category_name == ""Restaurant""\nRETURN b\n```",110.89,Incorrect - better schema names but wrong relations
3,Q001,mistral:latest,few_shot,"FOR b IN Businesses\n FILTER b.city == ""Texas""\n\n FOR c IN 1..1 OUTBOUND b BusinessCategory\n FILTER c.category_name == ""Restaurants"" AND c.sub_category_name == ""Moroccan""\n\n RETURN DISTINCT {\n business_id: b.business_id,\n name: b.name,\n city: b.city,\n rating: b.rating\n }",106.05,Incorrect - close to Gold but city/state confusion
4,Q001,qwen3:8b,direct,FOR r IN restaurants\nFILTER r.country == 'Morocco' AND r.state == 'Texas'\nRETURN r,257.68,Incorrect - schema hallucination
5,Q001,qwen3:8b,full_schema,"FOR business IN Businesses\n FILTER business.state == ""Texas""\n FILTER EXISTS(\n FOR category IN BusinessCategory\n FILTER category._from == business._id\n FILTER DOCUMENT(category._to).category_name == ""Moroccan""\n RETURN true\n )\n RETURN business",845.15,Execution failed - invalid use of EXISTS()
6,Q001,qwen3:8b,reduced_schema,FOR business IN Businesses\n FILTER business.state == 'Texas'\n FILTER EXISTS (\n FOR edge IN BusinessCategory\n FILTER edge._from == business._id\n LET category = DOCUMENT(edge._to)\n FILTER category.category_name == 'Moroccan'\n RETURN true\n )\n RETURN business,516.48,Execution failed - invalid use of EXISTS()
7,Q001,qwen3:8b,few_shot,"FOR b IN Businesses\n FILTER b.city == ""Texas""\n\n FOR c IN 1..1 OUTBOUND b BusinessCategory\n FILTER c.category_name == ""Moroccan""\n\n RETURN DISTINCT {\n business_id: b.business_id,\n name: b.name,\n city: b.city,\n rating: b.rating\n }",214.96,Almost correct - city used instead of state
8,Q001,gemma3:4b,direct,"```aql\nFOR doc IN restaurants\n FILTER doc.cuisine == ""Moroccan"" AND doc.state == ""Texas""\n RETURN doc\n```",29.63,Incorrect - hallucinated collection 'restaurants' and attribute 'cuisine'
9,Q001,gemma3:4b,full_schema,"```AQL\nFOR doc IN YelpDB.Businesses\n FILTER doc.city == ""Pittsburgh"" AND doc.state == ""Pennsylvania""\nRETURN doc\n```",156.03,"Incorrect - wrong collection reference, invented location and Moroccan category ignored"


In [83]:
def clean_aql_output(text):
    text = text.strip()

    if text.startswith("```aql"):
        text = text[len("```aql"):].strip()
    elif text.startswith("```AQL"):
        text = text[len("```AQL"):].strip()
    elif text.startswith("```"):
        text = text[3:].strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text

In [85]:
validation_result = validator.validate(
    gemma_few_shot_aql_clean
)

validation_result

{'valid': True,
 'errors': [],
 'used_collections': ['BusinessCategory', 'Businesses']}

In [86]:
execution_result = executor.execute(
    gemma_few_shot_aql_clean
)

print("Exécutée :", execution_result["executed"])
print("Erreur :", execution_result["error"])

if execution_result["results"] is not None:
    print("Nombre de résultats :", len(execution_result["results"]))
    print("Premiers résultats :")
    print(execution_result["results"][:5])

Exécutée : True
Erreur : None
Nombre de résultats : 0
Premiers résultats :
[]


In [87]:
from src.db_connector import get_database

In [88]:
db = get_database(
    host="http://localhost:8529",
    database="YelpDB",
    username="root",
    password="root"
)

In [89]:
print("Nom de la base :", db.name)
print("Collections :", [c["name"] for c in db.collections()])

Nom de la base : YelpDB
Collections : ['Tips', 'Users', 'BusinessNeighborhood', 'Categories', 'ReviewsBusiness', '_apps', '_queues', '_jobs', 'TipsBusiness', 'Checkins', 'WritesTip', 'Neighborhoods', 'WritesReview', 'BusinessCheckin', '_aqlfunctions', 'Businesses', '_graphs', '_appbundles', 'BusinessCategory', '_analyzers', 'Reviews', '_frontend']


In [90]:
gemma_few_shot_aql_clean = clean_aql_output(
    gemma_few_shot_aql
)

print(gemma_few_shot_aql_clean)

FOR b IN Businesses
    FILTER b.state == "Texas"

    FOR c IN 1..1 OUTBOUND b BusinessCategory
        FILTER c.category_name == "Moroccan"

        RETURN DISTINCT {
            business_id: b.business_id,
            name: b.name,
            city: b.city,
            rating: b.rating
        }


In [91]:
execution_result = executor.execute(
    gemma_few_shot_aql_clean
)

print("Exécutée :", execution_result["executed"])
print("Erreur :", execution_result["error"])

if execution_result["results"] is not None:
    print("Nombre de résultats :", len(execution_result["results"]))
    print("Premiers résultats :")
    print(execution_result["results"][:5])

Exécutée : True
Erreur : None
Nombre de résultats : 0
Premiers résultats :
[]


In [93]:
dangerous_execution = executor.execute(
    dangerous_query
)

print(dangerous_execution)

{'executed': False, 'validation': {'valid': False, 'errors': ['Forbidden operation(s): REMOVE'], 'used_collections': ['Businesses']}, 'results': None, 'error': 'Query rejected by validator.'}


In [94]:
from src.privacy_filter import PrivacyFilter

privacy_filter = PrivacyFilter()

In [95]:
privacy_result = privacy_filter.check(
    gemma_few_shot_aql_clean
)

privacy_result

{'allowed': True, 'sensitive_fields': [], 'reason': None}

In [96]:
sensitive_query = """
FOR b IN Businesses
    RETURN {
        name: b.name,
        full_address: b.full_address,
        latitude: b.latitude,
        longitude: b.longitude
    }
"""

privacy_filter.check(sensitive_query)

{'allowed': False,
 'sensitive_fields': ['full_address', 'latitude', 'longitude'],
 'reason': 'Sensitive field(s) detected.'}